In [ ]:
import os
import subprocess
import numpy as np
import pandas as pd

import pickle
from pathlib import Path

In [ ]:
#define global variables
##scratch directory 
##work directory
##work1 ; directory for file from past experiment

#persistent disk
data = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/data/2025-11-05_viral_disease_cohort"

#bucket
my_bucket = os.getenv('WORKSPACE_BUCKET')
bucket_data = "{my_bucket}/20251104infectiousdiseasephewasduplicate/data/2025-11-05_viral_disease_cohort"


In [ ]:
# This query represents dataset "viral disease" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
def get_viral_condition_concepts():
    
    dataset_48844012_condition_sql = """
        SELECT
            c_occurrence.person_id,
            c_occurrence.condition_concept_id,
            c_standard_concept.concept_name as standard_concept_name,
            c_standard_concept.concept_code as standard_concept_code,
            c_standard_concept.vocabulary_id as standard_vocabulary,
            c_occurrence.condition_start_datetime,
            c_occurrence.condition_end_datetime,
            c_occurrence.condition_type_concept_id,
            c_type.concept_name as condition_type_concept_name,
            c_occurrence.stop_reason,
            c_occurrence.visit_occurrence_id,
            visit.concept_name as visit_occurrence_concept_name,
            c_occurrence.condition_source_value,
            c_occurrence.condition_source_concept_id,
            c_source_concept.concept_name as source_concept_name,
            c_source_concept.concept_code as source_concept_code,
            c_source_concept.vocabulary_id as source_vocabulary,
            c_occurrence.condition_status_source_value,
            c_occurrence.condition_status_concept_id,
            c_status.concept_name as condition_status_concept_name 
        FROM
            ( SELECT
                * 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
            WHERE
                (
                    condition_concept_id IN (SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (440029)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1)
                )  
                AND (
                    c_occurrence.PERSON_ID IN (SELECT
                        distinct person_id  
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                    WHERE
                        cb_search_person.person_id IN (SELECT
                            person_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                        WHERE
                            has_ehr_data = 1 ) 
                        AND cb_search_person.person_id IN (SELECT
                            person_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                        WHERE
                            has_whole_genome_variant = 1 
                        UNION
                        DISTINCT SELECT
                            person_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                        WHERE
                            has_lr_whole_genome_variant = 1 
                        UNION
                        DISTINCT SELECT
                            person_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                        WHERE
                            has_array_data = 1 ) 
                        AND cb_search_person.person_id IN (SELECT
                            criteria.person_id 
                        FROM
                            (SELECT
                                DISTINCT person_id, entry_date, concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                (concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                    WHERE
                                        concept_id IN (440029)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) 
                                AND is_standard = 1 )) criteria ) )
                    )
                ) c_occurrence 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_standard_concept 
                    ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_type 
                    ON c_occurrence.condition_type_concept_id = c_type.concept_id 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                    ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.concept` visit 
                    ON v.visit_concept_id = visit.concept_id 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_source_concept 
                    ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
            LEFT JOIN
                `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_status 
                    ON c_occurrence.condition_status_concept_id = c_status.concept_id"""

    df = pd.read_gbq(
            dataset_48844012_condition_sql,
            dialect="standard",
            use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
            progress_bar_type="tqdm_notebook")

    return df

In [ ]:
#wrangle for unqiue concepts, only taking those with counts >= 100
def wrangle_viral_cond_concepts(df):
    
    # Group by the concept ID, then count the unique person_ids in each group
    # Group by both columns, count unique patients, and reset to a DataFrame
    patient_counts_df = df.groupby(['condition_concept_id', 'standard_concept_name'])['person_id'].nunique().reset_index(name='patient_count')


    # Now, sort the new DataFrame by the 'patient_count' column
    patient_counts_df_sorted = patient_counts_df.sort_values(by='patient_count', ascending=False)
    
    #filter
    patient_counts_df_sorted_and_filtered = patient_counts_df_sorted[patient_counts_df_sorted['patient_count'] >= 100]

    return patient_counts_df_sorted_and_filtered 

In [ ]:
def demographics_table():
    """
    Fetch person demographics rows for a single concept_id,
    using your original SQL structure and injecting concept_id directly.
    """
    dataset = os.environ["WORKSPACE_CDR"]

    demographics_sql  = f"""
    SELECT
        person.person_id,
        
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
   
        p_race_concept.concept_name as race,
    
        p_ethnicity_concept.concept_name as ethnicity,
    
        p_sex_at_birth_concept.concept_name as sex_at_birth,
      
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `{dataset}.person` person 
    LEFT JOIN
        `{dataset}.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `{dataset}.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `{dataset}.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `{dataset}.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `{dataset}.concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `{dataset}.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `{dataset}.cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `{dataset}.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 
            UNION
            DISTINCT SELECT
                person_id 
            FROM
                `{dataset}.cb_search_person` p 
            WHERE
                has_lr_whole_genome_variant = 1 
            UNION
            DISTINCT SELECT
                person_id 
            FROM
                `{dataset}.cb_search_person` p 
            WHERE
                has_array_data = 1 ) )"""


    demographics_df = pd.read_gbq(
        demographics_sql,
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook"
    )

    return demographics_df

In [ ]:
def socioeconomics_table():
    """
    Fetch socioeconomic observations mapped to ZIP-3 SES values,
    using your original SQL structure and injecting the dataset directly.
    """
    dataset = os.environ["WORKSPACE_CDR"]

    socioeconomic_sql = """
    SELECT
        observation.person_id,
        observation.observation_datetime,
        zip_code.zip3_as_string as zip_code,
        zip_code.fraction_assisted_income as assisted_income,

        zip_code.median_income,
        zip_code.fraction_no_health_ins as no_health_insurance,
        zip_code.fraction_poverty as poverty,
      
        zip_code.deprivation_index,
        zip_code.acs as american_community_survey_year 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.zip3_ses_map` zip_code 
    JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.observation` observation 
            ON CAST(SUBSTR(observation.value_as_string, 0, STRPOS(observation.value_as_string, '*') - 1) AS INT64) = zip_code.zip3  
    WHERE
        observation.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 
            UNION
            DISTINCT SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_lr_whole_genome_variant = 1 
            UNION
            DISTINCT SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_array_data = 1 ) ) 
            AND observation_source_concept_id = 1585250 
            AND observation.value_as_string NOT LIKE 'Res%'"""


    socioeconomic_df = pd.read_gbq(
        socioeconomic_sql,
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook"
    )

    return socioeconomic_df

In [ ]:
def get_condition_summary(concept_id):
    """
    Fetches per-patient summary for the specified condition_concept_id,
    applying Cohort Builder UI filters (EHR + genomics, observation window,
    flat-events, standard concepts), and returns a DataFrame with one row per
    patient including:
      - condition_concept_id
      - standard_concept_name, standard_vocabulary
      - first and last diagnosis date for that concept
      - condition_type_concept_name, visit_occurrence_concept_name from first occurrence
      - visits_for_concept: count of unique visit_occurrence_id for the concept
      - visits_all_concepts: count of unique visits across all conditions
      - concept_count_ehr: count of distinct condition concepts in EHR
    """
    dataset = os.environ["WORKSPACE_CDR"]
    sql = f"""
    WITH
      ehr_genomics_patients AS (
        SELECT DISTINCT person_id
        FROM `{dataset}.cb_search_person`
        WHERE has_ehr_data = 1
          AND (
               has_whole_genome_variant      = 1
            OR has_lr_whole_genome_variant   = 1
            OR has_array_data                = 1
          )
      ),

      all_occ AS (
        SELECT
          co.person_id,
          co.condition_concept_id,
          co.visit_occurrence_id,
          co.condition_start_datetime,
          co.condition_end_datetime,
          co.condition_type_concept_id
        FROM `{dataset}.condition_occurrence` co
        JOIN ehr_genomics_patients eg
          ON co.person_id = eg.person_id

        -- only events that made it into the CB search table
        JOIN `{dataset}.cb_search_all_events` ev
          ON ev.person_id = co.person_id
         AND ev.concept_id = co.condition_concept_id
         AND DATE(co.condition_start_datetime) = ev.entry_date

        JOIN `{dataset}.concept` c_std
          ON co.condition_concept_id = c_std.concept_id
        WHERE c_std.standard_concept = 'S'
      ),

      spec_occ AS (
        SELECT *
        FROM all_occ
        WHERE condition_concept_id = {concept_id}
      ),

      detail AS (
        SELECT
          person_id,
          condition_concept_id,
          condition_start_datetime AS first_diag_date,
          condition_end_datetime   AS first_end_date,
          condition_type_concept_id,
          visit_occurrence_id,
          ROW_NUMBER() OVER (PARTITION BY person_id ORDER BY condition_start_datetime) AS rn
        FROM spec_occ
      ),

      first_detail AS (
        SELECT
          d.person_id,
          d.condition_concept_id,
          d.first_diag_date,
          d.first_end_date,
          c_std.concept_name        AS standard_concept_name,
          c_std.vocabulary_id       AS standard_vocabulary,
          c_type.concept_name       AS condition_type_concept_name,
          vis_evt.concept_name      AS visit_occurrence_concept_name
        FROM detail d
        JOIN `{dataset}.concept` c_std
          ON d.condition_concept_id = c_std.concept_id
        LEFT JOIN `{dataset}.concept` c_type
          ON d.condition_type_concept_id = c_type.concept_id
        LEFT JOIN `{dataset}.visit_occurrence` v
          ON d.visit_occurrence_id = v.visit_occurrence_id
        LEFT JOIN `{dataset}.concept` vis_evt
          ON v.visit_concept_id = vis_evt.concept_id
        WHERE d.rn = 1
      ),

      spec_metrics AS (
        SELECT
          person_id,
          MIN(condition_start_datetime) AS first_diag_date,
          MAX(condition_start_datetime) AS last_diag_date,
          COUNT(DISTINCT visit_occurrence_id) AS visits_for_concept
        FROM spec_occ
        GROUP BY person_id
      ),

      allv AS (
        SELECT
          person_id,
          COUNT(DISTINCT visit_occurrence_id) AS visits_all_concepts
        FROM all_occ
        GROUP BY person_id
      ),

      conc AS (
        SELECT
          person_id,
          COUNT(DISTINCT condition_concept_id) AS concept_count_ehr
        FROM all_occ
        GROUP BY person_id
      )

    SELECT
      fd.person_id,
      fd.condition_concept_id,
      fd.standard_concept_name,
      fd.standard_vocabulary,
      fd.first_diag_date,
      sm.last_diag_date,
      fd.condition_type_concept_name,
      fd.visit_occurrence_concept_name,
      sm.visits_for_concept,
      av.visits_all_concepts,
      cc.concept_count_ehr
    FROM first_detail fd
    JOIN spec_metrics sm  ON fd.person_id = sm.person_id
    LEFT JOIN allv av       ON fd.person_id = av.person_id
    LEFT JOIN conc cc       ON fd.person_id = cc.person_id
    """

    df = pd.read_gbq(
        sql,
        project_id=os.environ.get("BIGQUERY_PROJECT"),
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook",
    )
    return df

In [ ]:
def merge_data_table(concepts_csv): 
    
    cohort_dict = {}
    
    #read in concepts.csv and grab needed variables
    viral_concept_df = pd.read_csv(concepts_csv)
    concept_list = viral_concept_df['condition_concept_id'].tolist()
    concept_name = viral_concept_df['standard_concept_name'].to_list()
    
    
    # call demo and socio function for table and drop duplicates by person_id: one-row-per-patient tables once
    demo = demographics_table().drop_duplicates('person_id')
    socio = socioeconomics_table().drop_duplicates('person_id')

    for concept_id, concept in zip(concept_list, concept_name):
    
        cond = get_condition_summary(concept_id)
        
        merged = (
            cond
              .merge(demo,  on='person_id', how='left')
              .merge(socio, on='person_id', how='left')
        )
        
        
        cohort_dict[(concept_id, concept)] = merged
       

    return cohort_dict

In [ ]:
#
def create_df_pkl(df, directory):

    # 2) Choose a workspace folder for persistence
    out_file = Path(directory)

    # 3) Save the entire dict in one go
    with open(out_file, 'wb') as f:
        pickle.dump(df, f)

    print(f"Saved {len(df)} DataFrames to {out_file}")
                    
    

In [ ]:
# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

def export_csv_to_WS_bucket(df, file_name):
    
    # Replace df with THE NAME OF YOUR DATAFRAME
    my_dataframe = df   

    # Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
    destination_filename = file_name

    ########################################################################
    ##
    ################# DON'T CHANGE FROM HERE ###############################
    ##
    ########################################################################

    # save dataframe in a csv file in the same workspace as the notebook
    my_dataframe.to_csv(destination_filename, index=False)

    # get the bucket name
    my_bucket = os.getenv('WORKSPACE_BUCKET')

    # copy csv file to the bucket
    args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/data/"]
    output = subprocess.run(args, capture_output=True)

    # print output from gsutil
    output.stderr


In [ ]:
# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

def export_pkl_to_WS_bucket(file_name):
    


    # Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
    destination_filename = file_name

    ########################################################################
    ##
    ################# DON'T CHANGE FROM HERE ###############################
    ##
    ########################################################################



    # get the bucket name
    my_bucket = os.getenv('WORKSPACE_BUCKET')

    # copy csv file to the bucket
    args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/data/"]
    output = subprocess.run(args, capture_output=True)

    # print output from gsutil
    output.stderr


In [ ]:
#function calls

#all_df = get_viral_condition_concepts()
#final_viral_concepts = wrangle_viral_cond_concepts(all_df)
merged_dict = merge_data_table(f"{data}/Final_cohort_query.csv")

In [ ]:
#csv, pkl, bucket export

#csv

#all_df.to_csv(f"{data}/full_viral_disease_cohort_query_dataset.csv")
#final_viral_concepts.to_csv(f"{data}/Final_cohort_query.csv")

#pkl

#create_df_pkl(all_df, f"{data}/full_viral_disease_cohort_query_dataset.pkl")
#create_df_pkl(merged_dict, f"{data}/viral_disease_cohort_summary_stats_dataset_dict.pkl")

#bucket

#export_csv_to_WS_bucket(all_df,"full_viral_disease_cohort_query_dataset.csv" )
#export_pkl_to_WS_bucket("viral_disease_cohort_summary_stats_dataset_dict.pkl" )
